![Snowflake](https://www.snowflake.com/wp-content/themes/flavor/assets/img/logo-snowflake-sans-word.svg)
# CRB VIP Comercios — Benchmark Transaccional
**RFP 10010806** · CredibanCo · Snowflake 2026

---

Este notebook crea la base de datos `CRB_VIP_COMERCIOS` con datos de benchmark transaccional por ciudad y categoría de comercio en Colombia. Incluye una función analítica que permite a cada comercio compararse contra su segmento.

## 1. Configuración inicial
Seleccionamos el rol, warehouse y creamos la base de datos con su schema.

In [ ]:
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE CREDIBANCO_HOL_WH;

CREATE DATABASE IF NOT EXISTS CRB_VIP_COMERCIOS;
USE DATABASE CRB_VIP_COMERCIOS;
CREATE SCHEMA IF NOT EXISTS ANALYTICS;
USE SCHEMA ANALYTICS;

## 2. Tabla de Benchmark por Categoría
Contiene promedios de facturación, ticket promedio, volumen de transacciones y distribución por percentiles para **5 ciudades × 8 categorías = 40 registros**.

In [ ]:
CREATE OR REPLACE TABLE BENCHMARK_CATEGORIA (
    CIUDAD                      VARCHAR,
    CATEGORIA_COMERCIO          VARCHAR,
    NUM_COMERCIOS               NUMBER,
    FACTURACION_PROMEDIO_MENSUAL NUMBER(15,2),
    TICKET_PROMEDIO             NUMBER(12,2),
    NUM_TRANSACCIONES_PROMEDIO  NUMBER,
    PERCENTIL_25                NUMBER(15,2),
    PERCENTIL_75                NUMBER(15,2),
    PERIODO                     VARCHAR
);

## 3. Cargar datos sintéticos
Datos agregados y anonimizados de benchmark transaccional para **Bogotá, Medellín, Cali, Barranquilla y Cartagena** en 8 categorías: Restaurantes, Retail Moda, Supermercados, Farmacias, Tecnología, Belleza, Salud y Entretenimiento.

In [ ]:
INSERT INTO BENCHMARK_CATEGORIA VALUES
('Bogotá','Restaurantes',4200,22000000.00,45000.00,489,12500000.00,30000000.00,'2024-Q3'),
('Bogotá','Retail Moda',1800,35000000.00,120000.00,292,18000000.00,48000000.00,'2024-Q3'),
('Bogotá','Supermercados',950,85000000.00,78000.00,1090,55000000.00,110000000.00,'2024-Q3'),
('Bogotá','Farmacias',1600,28000000.00,52000.00,538,16000000.00,38000000.00,'2024-Q3'),
('Bogotá','Tecnología',720,42000000.00,350000.00,120,22000000.00,58000000.00,'2024-Q3'),
('Bogotá','Belleza',2100,15000000.00,38000.00,395,8000000.00,21000000.00,'2024-Q3'),
('Bogotá','Salud',1100,48000000.00,95000.00,505,28000000.00,65000000.00,'2024-Q3'),
('Bogotá','Entretenimiento',650,32000000.00,65000.00,492,17000000.00,44000000.00,'2024-Q3'),
('Medellín','Restaurantes',2800,19000000.00,42000.00,452,10500000.00,26000000.00,'2024-Q3'),
('Medellín','Retail Moda',1200,30000000.00,110000.00,273,15000000.00,42000000.00,'2024-Q3'),
('Medellín','Supermercados',680,72000000.00,72000.00,1000,48000000.00,92000000.00,'2024-Q3'),
('Medellín','Farmacias',1050,24000000.00,48000.00,500,14000000.00,33000000.00,'2024-Q3'),
('Medellín','Tecnología',480,38000000.00,320000.00,119,19000000.00,52000000.00,'2024-Q3'),
('Medellín','Belleza',1500,13000000.00,35000.00,371,7000000.00,18000000.00,'2024-Q3'),
('Medellín','Salud',750,44000000.00,88000.00,500,25000000.00,60000000.00,'2024-Q3'),
('Medellín','Entretenimiento',420,28000000.00,58000.00,483,14000000.00,39000000.00,'2024-Q3'),
('Cali','Restaurantes',1900,17000000.00,40000.00,425,9000000.00,24000000.00,'2024-Q3'),
('Cali','Retail Moda',850,26000000.00,105000.00,248,13000000.00,36000000.00,'2024-Q3'),
('Cali','Supermercados',520,65000000.00,68000.00,956,42000000.00,85000000.00,'2024-Q3'),
('Cali','Farmacias',780,21000000.00,45000.00,467,12000000.00,29000000.00,'2024-Q3'),
('Cali','Tecnología',320,34000000.00,290000.00,117,17000000.00,47000000.00,'2024-Q3'),
('Cali','Belleza',1100,12000000.00,33000.00,364,6500000.00,17000000.00,'2024-Q3'),
('Cali','Salud',580,40000000.00,82000.00,488,22000000.00,55000000.00,'2024-Q3'),
('Cali','Entretenimiento',310,25000000.00,55000.00,455,12000000.00,35000000.00,'2024-Q3'),
('Barranquilla','Restaurantes',1400,16000000.00,38000.00,421,8500000.00,22000000.00,'2024-Q3'),
('Barranquilla','Retail Moda',620,24000000.00,98000.00,245,12000000.00,34000000.00,'2024-Q3'),
('Barranquilla','Supermercados',380,58000000.00,65000.00,892,38000000.00,75000000.00,'2024-Q3'),
('Barranquilla','Farmacias',560,19000000.00,42000.00,452,10000000.00,26000000.00,'2024-Q3'),
('Barranquilla','Tecnología',240,30000000.00,270000.00,111,15000000.00,42000000.00,'2024-Q3'),
('Barranquilla','Belleza',820,11000000.00,30000.00,367,6000000.00,15000000.00,'2024-Q3'),
('Barranquilla','Salud',420,37000000.00,78000.00,474,20000000.00,50000000.00,'2024-Q3'),
('Barranquilla','Entretenimiento',230,22000000.00,50000.00,440,11000000.00,31000000.00,'2024-Q3'),
('Cartagena','Restaurantes',1100,18000000.00,48000.00,375,9500000.00,25000000.00,'2024-Q3'),
('Cartagena','Retail Moda',480,27000000.00,115000.00,235,14000000.00,38000000.00,'2024-Q3'),
('Cartagena','Supermercados',290,52000000.00,62000.00,839,34000000.00,68000000.00,'2024-Q3'),
('Cartagena','Farmacias',420,20000000.00,44000.00,455,11000000.00,28000000.00,'2024-Q3'),
('Cartagena','Tecnología',180,31000000.00,280000.00,111,16000000.00,44000000.00,'2024-Q3'),
('Cartagena','Belleza',650,12500000.00,34000.00,368,7000000.00,17500000.00,'2024-Q3'),
('Cartagena','Salud',350,38000000.00,80000.00,475,21000000.00,52000000.00,'2024-Q3'),
('Cartagena','Entretenimiento',280,26000000.00,60000.00,433,13000000.00,36000000.00,'2024-Q3');


## 4. Verificar datos cargados
Confirmamos que los 40 registros se insertaron correctamente y exploramos el benchmark de Bogotá.

In [ ]:
-- Conteo total
SELECT COUNT(*) AS TOTAL_REGISTROS FROM BENCHMARK_CATEGORIA;


In [ ]:
-- Benchmark de Bogotá por categoría
SELECT 
    CATEGORIA_COMERCIO,
    NUM_COMERCIOS,
    TO_CHAR(FACTURACION_PROMEDIO_MENSUAL, '$999,999,999,999') AS FACTURACION_PROMEDIO,
    TO_CHAR(TICKET_PROMEDIO, '$999,999,999') AS TICKET_PROMEDIO,
    NUM_TRANSACCIONES_PROMEDIO
FROM BENCHMARK_CATEGORIA
WHERE CIUDAD = 'Bogotá'
ORDER BY FACTURACION_PROMEDIO_MENSUAL DESC;

## 5. Función de comparación: FN_COMPARAR_COMERCIO
Esta función recibe **ciudad**, **categoría** y **facturación mensual** de un comercio y retorna:
- **Posición** relativa (MUY POR ENCIMA / POR ENCIMA / POR DEBAJO / MUY POR DEBAJO)
- **Percentil estimado** dentro de su categoría
- **Recomendación** accionable personalizada

In [ ]:
CREATE OR REPLACE FUNCTION ANALYTICS.FN_COMPARAR_COMERCIO(
    P_CIUDAD VARCHAR, P_CATEGORIA VARCHAR, P_MI_FACTURACION NUMBER
) RETURNS TABLE (
    CIUDAD VARCHAR, CATEGORIA VARCHAR, MI_FACTURACION NUMBER(15,2),
    PROMEDIO_CATEGORIA NUMBER(15,2), DIFERENCIA_PESOS NUMBER(15,2),
    DIFERENCIA_PCT NUMBER(5,1), POSICION VARCHAR, PERCENTIL_ESTIMADO NUMBER,
    RECOMENDACION VARCHAR
) AS $$
    SELECT
        b.CIUDAD, b.CATEGORIA_COMERCIO, P_MI_FACTURACION,
        b.FACTURACION_PROMEDIO_MENSUAL,
        P_MI_FACTURACION - b.FACTURACION_PROMEDIO_MENSUAL,
        ROUND((P_MI_FACTURACION - b.FACTURACION_PROMEDIO_MENSUAL) / NULLIF(b.FACTURACION_PROMEDIO_MENSUAL, 0) * 100, 1),
        CASE
            WHEN P_MI_FACTURACION > b.PERCENTIL_75 THEN 'MUY POR ENCIMA'
            WHEN P_MI_FACTURACION > b.FACTURACION_PROMEDIO_MENSUAL THEN 'POR ENCIMA'
            WHEN P_MI_FACTURACION > b.PERCENTIL_25 THEN 'POR DEBAJO'
            ELSE 'MUY POR DEBAJO'
        END,
        CASE
            WHEN P_MI_FACTURACION >= b.PERCENTIL_75 THEN 75 + LEAST(25, ROUND((P_MI_FACTURACION - b.PERCENTIL_75) / NULLIF(b.PERCENTIL_75, 0) * 50, 0))
            WHEN P_MI_FACTURACION >= b.FACTURACION_PROMEDIO_MENSUAL THEN 50 + ROUND((P_MI_FACTURACION - b.FACTURACION_PROMEDIO_MENSUAL) / NULLIF(b.PERCENTIL_75 - b.FACTURACION_PROMEDIO_MENSUAL, 0) * 25, 0)
            WHEN P_MI_FACTURACION >= b.PERCENTIL_25 THEN 25 + ROUND((P_MI_FACTURACION - b.PERCENTIL_25) / NULLIF(b.FACTURACION_PROMEDIO_MENSUAL - b.PERCENTIL_25, 0) * 25, 0)
            ELSE GREATEST(1, ROUND(P_MI_FACTURACION / NULLIF(b.PERCENTIL_25, 0) * 25, 0))
        END,
        CASE
            WHEN P_MI_FACTURACION > b.PERCENTIL_75 THEN 'Excelente. Tu comercio de ' || b.CATEGORIA_COMERCIO || ' en ' || b.CIUDAD || ' esta en el top 25%. Considera expandir o abrir otra sede.'
            WHEN P_MI_FACTURACION > b.FACTURACION_PROMEDIO_MENSUAL THEN 'Bien. Estas por encima del promedio de ' || b.CATEGORIA_COMERCIO || ' en ' || b.CIUDAD || '. Oportunidad de escalar con campanas de fidelizacion.'
            WHEN P_MI_FACTURACION > b.PERCENTIL_25 THEN 'Tu facturacion esta por debajo del promedio. Revisa ticket promedio y frecuencia de compra para ' || b.CATEGORIA_COMERCIO || ' en ' || b.CIUDAD || '.'
            ELSE 'Alerta: estas en el cuartil inferior de ' || b.CATEGORIA_COMERCIO || ' en ' || b.CIUDAD || '. Recomendamos una revision integral de estrategia comercial.'
        END
    FROM BENCHMARK_CATEGORIA b
    WHERE b.CIUDAD = P_CIUDAD AND b.CATEGORIA_COMERCIO = P_CATEGORIA
$$;

## 6. Pruebas — Compara tu comercio
Ejecuta estos ejemplos para ver cómo se compara un comercio contra su categoría.

In [ ]:
-- Ejemplo 1: Restaurante en Bogotá que factura $18M (por debajo del promedio de $22M)
SELECT * FROM TABLE(ANALYTICS.FN_COMPARAR_COMERCIO('Bogotá', 'Restaurantes', 18000000));

In [ ]:
-- Ejemplo 2: Farmacia en Medellín que factura $35M (por encima del promedio de $24M)
SELECT * FROM TABLE(ANALYTICS.FN_COMPARAR_COMERCIO('Medellín', 'Farmacias', 35000000));

In [ ]:
-- Ejemplo 3: Comparar múltiples ciudades para una farmacia de $30M
-- "Estoy evaluando abrir una farmacia: ¿en qué ciudad me iría mejor?"
SELECT * FROM TABLE(ANALYTICS.FN_COMPARAR_COMERCIO('Bogotá', 'Farmacias', 30000000))
UNION ALL
SELECT * FROM TABLE(ANALYTICS.FN_COMPARAR_COMERCIO('Medellín', 'Farmacias', 30000000))
UNION ALL
SELECT * FROM TABLE(ANALYTICS.FN_COMPARAR_COMERCIO('Cali', 'Farmacias', 30000000))
UNION ALL
SELECT * FROM TABLE(ANALYTICS.FN_COMPARAR_COMERCIO('Barranquilla', 'Farmacias', 30000000))
UNION ALL
SELECT * FROM TABLE(ANALYTICS.FN_COMPARAR_COMERCIO('Cartagena', 'Farmacias', 30000000));

## ✅ Listo
La base de datos `CRB_VIP_COMERCIOS` está creada con:
- **Tabla:** `ANALYTICS.BENCHMARK_CATEGORIA` — 40 registros (5 ciudades × 8 categorías)
- **Función:** `ANALYTICS.FN_COMPARAR_COMERCIO(ciudad, categoría, facturación)` — diagnóstico automático

Estos objetos se incluyen en el **Listing** del ejercicio de monetización.